In [1]:
import pandas as pd

MONGO_URI = "mongodb://10.3.32.15:27017/"
MONGO_DB = "main"
COLLECTION = "tracking_sessions_v3_b"

In [2]:
path = "buildingID_to_uuid.csv"
df = pd.read_csv(path)
df

,BuildingID,UUID
0,69351bbd5a54c51c3d607e34,097d333e-44ff-4eca-b243-eb8e2e9ea453
1,69351bbd5a54c51c3d607e34,217048d5-97ce-48f3-b6da-02c7c92b1d64
2,69351bbd5a54c51c3d607e34,2a57d46a-c552-4b6e-ba09-31e5f44b0279
3,69351bbd5a54c51c3d607e34,396c1952-d233-46c6-80d8-ad67b4e263d8
4,69351bbd5a54c51c3d607e34,3b469195-68a1-4cf9-8b27-e1bc3252311b
...,...,...
24528,69351c010c5368ae30aca230,e9f093cb-f034-4a26-92c0-f8eaac6543c2
24529,69351c010c5368ae30aca230,ff4c1cd6-9d8d-42e8-9cfc-497036093543
24530,69351c010c5368ae30aca230,5e2ab99a-ea1b-4c12-a6c9-3aa31da9d33b
24531,69351b2d9f0dae4df1d6622f,03ff73c8-4b55-4978-8af3-3a6b499233b1


In [3]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[COLLECTION]

print(f"Connected to {MONGO_DB}.{COLLECTION}")
print(f"Collection has {collection.count_documents({})} documents")

Connected to main.tracking_sessions_v3_b
Collection has 18249 documents


In [4]:
# Get all UUIDs from the collection
uuids_in_collection = set()
for doc in collection.find({}, {"uuid": 1}):
    if "uuid" in doc:
        uuids_in_collection.add(doc["uuid"])

print(f"Found {len(uuids_in_collection)} UUIDs in collection")

Found 17162 UUIDs in collection


In [5]:
# Find UUIDs NOT in the collection
df['in_collection'] = df['UUID'].isin(uuids_in_collection)
uuids_not_in_collection = df[~df['in_collection']]

print(f"UUIDs NOT in collection: {len(uuids_not_in_collection)}")
print(f"UUIDs in collection: {len(df[df['in_collection']])}")

UUIDs NOT in collection: 12358
UUIDs in collection: 12175


In [6]:
# Save the UUIDs not in collection to a CSV
output_path = "uuids_not_in_collection.csv"
uuids_not_in_collection[['BuildingID', 'UUID']].to_csv(output_path, index=False)

print(f"Saved {len(uuids_not_in_collection)} UUIDs to {output_path}")

Saved 12358 UUIDs to uuids_not_in_collection.csv
